## Deep L-layer Neural Network
### 1. The Concept of Neural Network Depth
After getting familiar with foundational concepts like forward propagation, backpropagation, vectorization, and random initialization, the next step is to expand the architecture from simple to more complex models.
- **Shallow Models:** Logistic Regression is considered the shallowest model (equivalent to a 1-layer neural network). A neural network with 1 hidden layer is called a 2-layer network (we do not count the input layer when counting layers).
- **Deep Models:** These are neural networks with multiple hidden layers (e.g., 2 layers, 5 layers, or dozens of layers).
- **Why do we need deep networks?** Reality in the Artificial Intelligence community shows that there are complex mathematical functions and data features that only very deep neural networks have the capacity to learn, while shallower models often fail.
- **Choosing the depth:** There is no fixed formula to know in advance exactly how deep of a neural network a problem requires. The number of hidden layers is a hyperparameter. The standard process is to experiment starting with a simple model (Logistic Regression), then gradually increase the number of hidden layers (1, 2, etc.) and evaluate the performance on a validation/dev set to choose the optimal architecture.

<img src="../images/Week04/image1.png" width=800>

### 2. Mathematical Notation System for Deep Neural Networks
To process calculations on a multi-layer network, we need a consistent and clear notation system. Suppose we have a 4-layer neural network (3 hidden layers and 1 output layer).

The core notations include:
- **$L$:** The total number of layers in the neural network (excluding the input layer).
- **$l$:** The index of a specific layer, ranging from $1$ to $L$. The input layer is indexed as $0$.
- **$n^{[l]}$:** The number of neurons (units) in layer $l$.
  - The input layer is denoted as $n^{[0]} = n_x$ (the number of data features).
  - The output layer is denoted as $n^{[L]}$ (usually equal to 1 for binary classification problems).
- **$a^{[l]}$:** The activation matrix at layer $l$.
  - At the input layer, the activation value is exactly the original data: $a^{[0]} = X$.
  - At the final layer, the activation value is exactly the prediction result: $a^{[L]} = \hat{Y}$.
- **$W^{[l]}$ and $b^{[l]}$:** The weight matrix and bias vector used to compute the linear matrix $Z^{[l]}$ at layer $l$.
- **$g^{[l]}$:** The activation function applied specifically to layer $l$ (e.g., applying ReLU for hidden layers and Sigmoid for the output layer). The general equation is $a^{[l]} = g^{[l]}(Z^{[l]})$.

This notation system will accompany you throughout the process of programming Deep Learning algorithms, helping to avoid confusion about the structure and dimensions of the matrix space when performing backpropagation calculations.

<img src="../images/Week04/image2.png" width=800>

## Forward Propagation in a Deep Network
### 1. Forward Propagation for a Single Data Sample ($x$)
When passing a single data point $x$ (a column vector) through an $L$-layer deep neural network, the computation process occurs sequentially from the first hidden layer to the output layer.

At each layer $l$ (with $l$ ranging from $1$ to $L$), the system performs two computational steps:
- **Calculate the linear sum ($z^{[l]}$):** Multiply the weight matrix $W^{[l]}$ by the activation value of the previous layer $a^{[l-1]}$, then add the bias vector $b^{[l]}$. $$z^{[l]} = W^{[l]} a^{[l-1]} + b^{[l]}$$
- **Apply the activation function ($a^{[l]}$):** Pass the newly calculated $z^{[l]}$ value through a non-linear activation function $g^{[l]}$ (such as ReLU, Sigmoid, or Tanh) to generate the output signal of the current layer. $$a^{[l]} = g^{[l]}(z^{[l]})$$

**Important convention:** The input data $x$ is exactly the activation value of layer 0. That is, $a^{[0]} = x$. Thanks to this convention, the set of equations above becomes completely consistent and applicable to every layer in the network. The result at the final layer $a^{[L]}$ is exactly the prediction value $\hat{y}$.

### 2. Vectorizing Forward Propagation for the Entire Dataset ($m$ samples)
To fully leverage the parallel computing power of hardware (like GPUs) and avoid using time-consuming for loops to iterate through each data sample, we vectorize the entire training set consisting of $m$ samples.

Method:
- Stack the $m$ input vectors $x^{(1)}, x^{(2)}, \dots, x^{(m)}$ horizontally to form the input matrix $X$. By the new convention, $X$ is exactly the matrix $A^{[0]}$.
- Similarly, the $z^{[l]}$ and $a^{[l]}$ vectors of each sample will be stacked side-by-side into large matrices $Z^{[l]}$ and $A^{[l]}$.

In this case, the vectorized forward propagation equations at layer $l$ will be:
$$
Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}
$$
$$
A^{[l]} = g^{[l]}(Z^{[l]})
$$

This process continues until the final layer, returning the matrix $A^{[L]}$ containing all predictions $\hat{Y}$ for the $m$ data samples stacked horizontally.

### 3. A Note on For Loops in Deep Neural Networks
One of the golden rules when programming Machine Learning is to try to eliminate explicit for loops through vectorization. However, this rule only applies to the data dimension (iterating over $m$ samples).

For the depth dimension of the network (iterating over $L$ hidden layers), you **must use an explicit** for loop from $l = 1$ to $L$. The reason is that the neural network architecture is strictly sequential: You cannot compute the activation of layer 3 without having completed the result of layer 2.

Therefore, a standard code snippet to run forward propagation will look like this:
```python
A = X
for l in range(1, L + 1):
    Z = np.dot(W[l], A) + b[l]
    A = g[l](Z)
```

<img src="../images/Week04/image3.png" width=800>

## Getting your Matrix Dimensions Right
## 1. The Importance of Dimensionality Checking
When programming Deep Neural Networks, tracking dozens of variables like $W, b, Z, A$ across multiple hidden layers can easily lead to invalid matrix multiplication errors (Value Error / Shape Error).

The secret to writing bug-free code is: Always take a piece of scratch paper, clearly outline the network architecture, and write down the exact dimensions (number of rows, number of columns) of each matrix at each layer before typing your first line of code. If the matrix dimensions match perfectly according to linear algebra rules, your algorithm will almost certainly run correctly.

As an illustrative example, consider a 5-layer neural network ($L=5$) performing a binary classification problem with the following architecture:
- Input layer ($l=0$): $n^{[0]} = n_x = 2$ features.
- Hidden layer 1 ($l=1$): $n^{[1]} = 3$ neurons.
- Hidden layer 2 ($l=2$): $n^{[2]} = 5$ neurons.
- Hidden layer 3 ($l=3$): $n^{[3]} = 4$ neurons.
- Hidden layer 4 ($l=4$): $n^{[4]} = 2$ neurons.
- Output layer ($l=5$): $n^{[5]} = 1$ neuron.

### 2. Controlling the Dimensions of Weights ($W$) and Biases ($b$)
The parameters $W$ and $b$ are static matrices; their dimensions never change regardless of whether you are training on a single data sample or $m$ data samples.

Recall the fundamental linear equation at layer $l$:
$$
z^{[l]} = W^{[l]} a^{[l-1]} + b^{[l]}
$$
From this equation, we can deduce the rules for setting up dimensions:
- Weight Matrix $W^{[l]}$
  - The function of $W^{[l]}$ is to transform data from the $n^{[l-1]}$-dimensional space (of the previous layer) to the $n^{[l]}$-dimensional space (of the current layer).
  - Therefore, the exact dimensions of $W^{[l]}$ must be: $(n^{[l]}, n^{[l-1]})$
  - Example: $W^{[1]}$ has dimensions $(3, 2)$; $W^{[2]}$ has dimensions $(5, 3)$; $W^{[5]}$ has dimensions $(1, 2)$.
- Bias Vector $b^{[l]}$
  - The function of $b^{[l]}$ is to add a bias value to each individual neuron in the current layer. Thus, the number of elements in $b$ must exactly equal the number of neurons in that layer.
  - The exact dimensions of $b^{[l]}$ must be a column vector: $(n^{[l]}, 1)$
  - Example: $b^{[1]}$ has dimensions $(3, 1)$; $b^{[2]}$ has dimensions $(5, 1)$.

- Gradient Matrices (Gradients)

  During Backpropagation, the ultimate rule is that the gradient matrix must always have the exact same structure as its original matrix:
  - $dW^{[l]}$ has the same dimensions as $W^{[l]}$: $(n^{[l]}, n^{[l-1]})$
  - $db^{[l]}$ has the same dimensions as $b^{[l]}$: $(n^{[l]}, 1)$

<img src="../images/Week04/image4.png" width=800>

### 3. Controlling the Dimensions of Data ($Z$ and $A$)
In contrast to $W$ and $b$, the dimensions of the matrices containing the computed data ($Z, A$) will change depending on whether you are processing a single data sample or vectorizing across $m$ data samples.

**Case 1: Training on a single data sample ($x$)**
- $x$ is a column vector with dimensions $(n^{[0]}, 1)$.
- $z^{[l]}$ and $a^{[l]}$ are also column vectors containing the values of each neuron in the layer.
- The dimensions of $z^{[l]}$ and $a^{[l]}$ are: $(n^{[l]}, 1)$

**Case 2: Implementing Vectorization on the entire dataset ($m$ samples)**

When stacking $m$ data samples horizontally side-by-side, $X$ becomes a large matrix. The generated $z$ and $a$ values also automatically expand into matrices $Z$ and $A$ with $m$ columns.
- The input matrix $X$ (equivalent to $A^{[0]}$) has dimensions: $(n^{[0]}, m)$
- The linear matrix $Z^{[l]}$ and activation matrix $A^{[l]}$ have dimensions: $(n^{[l]}, m)$
- Similar to the derivative rule, $dZ^{[l]}$ and $dA^{[l]}$ also have dimensions: $(n^{[l]}, m)$

**Note on the calculation $Z^{[1]} = W^{[1]} X + b^{[1]}$:**
- $W^{[1]}$ is an $(n^{[1]}, n^{[0]})$ matrix.
- $X$ is an $(n^{[0]}, m)$ matrix.
- The result of the matrix multiplication $W^{[1]} X$ will be an $(n^{[1]}, m)$ matrix.
- The variable $b^{[1]}$ is just an $(n^{[1]}, 1)$ vector. However, thanks to Python/NumPy's **Broadcasting** mechanism, the vector $b^{[1]}$ will automatically be duplicated (broadcasted) into $m$ columns to perfectly match the $(n^{[1]}, m)$ dimensions before the element-wise addition is performed.

<img src="../images/Week04/image5.png" width=800>

## Why Deep Representations?
### 1. Intuition on Hierarchical Representation
The most practical reason why deep neural networks perform exceptionally well lies in their ability to extract features in a hierarchical structure, from simple to complex. Neural networks learn to understand the world in a way similar to how the human brain synthesizes information.

- **Example in Computer Vision (Facial Recognition)**
  
  When feeding an original image (a collection of pixels) into a deep neural network:

  - **Early layers (Shallow):** Function as edge detectors. They only learn to detect light and dark bands, horizontal edges, vertical edges, or simple corners.
  - **Middle layers:** Aggregate the detected edges to recognize more complex parts, such as eyes, noses, mouths, or ears.
  - **Final layers (Deep):** Combine the parts together to form overall templates, allowing the network to accurately recognize specific faces.

- Example in Audio Processing (Speech Recognition)
  
  When feeding an audio wave into the network:

  - **Early layers:** Learn to detect basic physical properties of the sound wave, such as rising and falling pitch or noise frequencies.
  - **Middle layers:** Group waveform properties to detect phonemes, the most basic sound units that make up words (for example, the sounds c, a, t).
  - **Final layers:** Assemble phonemes into complete words, then combine words into meaningful phrases and sentences.

**Conclusion:** Stacking multiple layers allows the neural network to reuse simple concepts at lower layers to construct complex concepts at higher layers. A shallow network would have to try to learn directly from pixels to faces, which is an incredibly difficult task.

<img src="../images/Week04/image6.png" width=800>

### 2. Mathematical Foundation from Circuit Theory
Computational Circuit Theory provides a rigorous mathematical proof to answer the question: If I create a neural network with only 1 hidden layer but give it an infinite number of neurons, will it be equivalent to a deep network?

The answer is: **It is possible, but extremely inefficient and resource-intensive**.

Take the Exclusive OR (XOR) calculation for $n$ input variables as an example.
- **If using a Deep Network (Hierarchical structure):** You can group pairs of variables to compute the XOR, then continue grouping the results in a tree structure.
  - The depth of the network only needs to grow logarithmically: $\mathcal{O}(\log n)$.
  - The number of required neurons (logic gates) grows linearly: $\mathcal{O}(n)$.
  - This is an incredibly streamlined and optimal architecture.
- **If using a Shallow Network (Only 1 single hidden layer):** The network cannot aggregate information step-by-step. Instead, this single hidden layer is forced to memorize every possible combination of the $n$ input variables.
  - The number of required neurons will explode exponentially: $\mathcal{O}(2^n)$ or more precisely $2^{n-1}$.
  - If $n$ is large, the system will collapse because the massive memory and computational requirements cannot be met.

**Conclusion:** There are complex mathematical functions that deep networks can solve with a small number of neurons thanks to their hierarchical structure. To compute the exact same function, a shallow network would require an exponentially large number of neurons.

<img src="../images/Week04/image7.png" width=800>

### 3. Practical Advice on Model Building
- The term Deep Learning was initially a highly successful rebranding strategy for the concept of artificial neural networks with multiple hidden layers. Despite its PR nature, the actual power of deep networks today is undeniable, especially when networks can reach tens or hundreds of layers.
- However, not every problem requires a deep network. When starting to solve a new problem, Professor Andrew Ng advises always starting with Logistic Regression (0 hidden layers), then gradually increasing to 1 hidden layer, 2 hidden layers.
- The number of layers (depth) should be treated as a Hyperparameter. You need to continuously adjust and cross-reference the results on the Cross-Validation set to find the sufficient and most optimal depth for your data. Avoid overusing excessively deep networks, which leads to time-consuming training and the risk of Overfitting.

## Building Blocks of Deep Neural Networks
### 1. Forward Propagation Block
At any given hidden layer $l$ in the neural network, the forward propagation block acts as a data processing function taking inputs to generate outputs for the next layer.
- **Input:** Receives the activation matrix $a^{[l-1]}$ from the immediately preceding layer.
- **Parameters used:** Uses the weight matrix $W^{[l]}$ and bias vector $b^{[l]}$ of that specific layer.
- **Calculations:**
  - **Computes the linear sum:** $Z^{[l]} = W^{[l]}a^{[l-1]} + b^{[l]}$
  - **Applies the activation function:** $a^{[l]} = g(Z^{[l]})$
- **Output:** Returns the activation matrix $a^{[l]}$ to pass forward into layer $l+1$.
- **Cache:** This is an extremely crucial step for practical programming. Layer $l$ will store the variable $Z^{[l]}$ (and typically $W^{[l]}$ and $b^{[l]}$ as well) into a cache. These values will wait there to serve the derivative calculations in the reverse direction.

### 2. Backward Propagation Block
When the neural network reaches the final layer and starts moving backward to correct errors, the backward propagation block at layer $l$ will receive the error and distribute it.
- **Input:** Receives the derivative matrix $da^{[l]}$ passed back from layer $l+1$.
- **Retrieved data:** Recalls the values $Z^{[l]}$, $W^{[l]}$, and $b^{[l]}$ that were previously stored in the Cache during the forward propagation step.
- **Calculations:** Computes the internal linear error amount $dZ^{[l]}$ of the layer.
- **Outputs:** 
  - Outputs the derivatives $dW^{[l]}$ and $db^{[l]}$ for the algorithm to update the parameters of this layer.
  - Outputs the derivative matrix $da^{[l-1]}$ to push back to layer $l-1$ in front of it to continue the calculations.

<img src="../images/Week04/image8.png" width=800>

### 3. The Comprehensive Cycle of a Training Iteration
The combination of the functional blocks above creates a complete training iteration for the neural network:
- **Forward pass:** Starts by feeding the original data $X$ (which is $a^{[0]}$) into the network. The data sequentially passes through the forward propagation functions from layer 1, 2, 3... up to the final layer $L$, continuously saving information into the Caches, and ends by outputting the prediction $\hat{y}$ (which is $a^{[L]}$).
- **Error calculation:** Uses the prediction $\hat{y}$ compared against the actual label to trigger the backpropagation process.
- **Backward pass:** The backpropagation chain starts running backward from layer $L$ to layer 1. At each station, it pulls information from the corresponding Cache, computes and releases the gradient sets ($dW, db$), while simultaneously passing the remaining error ($da$) back to the previous station.
- **Update:** Once all the gradients $dW$ and $db$ are fully available for every layer, the Gradient Descent algorithm will execute the update of the entire weight system based on the learning rate.

One such loop completes one instance of the model self-adjusting. The neural network will repeat this cycle thousands of times to continuously refine its parameters.

<img src="../images/Week04/image9.png" width=800>

## Forward and Backward Propagation
### 1. Forward Function
The forward propagation function at any given layer $l$ is responsible for receiving activation data from the previous layer ($A^{[l-1]}$), processing it through weight matrices, and outputting the activation data for the next layer ($A^{[l]}$) along with a Cache.

The vectorized equations for this function include:
- **Linear computation:** $Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}$
  
  (Note: Python will automatically apply the Broadcasting mechanism to the vector $b^{[l]}$ to correctly add it to the matrix $Z$).
- **Non-linear activation:** $A^{[l]} = g^{[l]}(Z^{[l]})$

  (Where $g^{[l]}$ can be ReLU for hidden layers, or Sigmoid for the output layer).
- **Cache storage:** Save $Z^{[l]}$ (as well as $W^{[l]}$ and $b^{[l]}$ for programming convenience) into a cache to be used for the backward cycle.

**Cycle initialization:** At the first layer ($l=1$), the input data $A^{[0]}$ is exactly the initial feature matrix $X$.

<img src="../images/Week04/image10.png" width=800>

### 2. Backward Function
The backward propagation function at layer $l$ is responsible for receiving the derivative from the subsequent layer ($dA^{[l]}$), retrieving data from the Cache ($Z^{[l]}, W^{[l]}$), and outputting the update derivatives ($dW^{[l]}, db^{[l]}$) along with the backpropagated derivative ($dA^{[l-1]}$).

This is the center of complex matrix calculus operations. The vectorized equations include:
- **Compute internal linear derivative:** $dZ^{[l]} = dA^{[l]} * g^{[l]'}(Z^{[l]})$

  (Element-wise multiplication between the backpropagated derivative and the derivative of the activation function at that layer).
- **Compute weight matrix derivative $W$:** $dW^{[l]} = \frac{1}{m} dZ^{[l]} (A^{[l-1]})^T$
- **Compute bias vector derivative $b$:** $db^{[l]} = \frac{1}{m} \text{np.sum}(dZ^{[l]}, \text{axis}=1, \text{keepdims=True})$
- **Compute derivative to backpropagate to the previous layer:** $dA^{[l-1]} = (W^{[l]})^T dZ^{[l]}$

Backward cycle initialization: The backpropagation process starts from the final layer $L$. To begin, you need to calculate the derivative of the Loss Function with respect to the prediction variable $\hat{Y}$ (which is exactly $A^{[L]}$).

For a binary classification problem using the Cross-Entropy loss function, the vectorized initialization equation is:
$$
dA^{[L]} = -\frac{Y}{A^{[L]}} + \frac{1 - Y}{1 - A^{[L]}}
$$
(The division here is performed element-wise between the actual label matrix $Y$ and the prediction matrix $A^{[L]}$).

<img src="../images/Week04/image11.png" width=800>

### 3. Practical Philosophy of Deep Learning Programming
Professor Andrew Ng shares a practical perspective on programming Machine Learning systems:
- **Initial confusion is normal:** Looking at a series of matrix calculus derivative equations ($dW, dA, dZ$) can feel overwhelming and abstract. The best way to understand them is not to try to memorize them, but to dive into practical programming exercises. When you manually type each line of code and cross-check matrix dimensions, the abstraction transforms into concrete understanding.
- **Power comes from data, not source code:** Even for seasoned experts, witnessing a successful machine learning model can sometimes still feel magical. A deep neural network does not require you to write hundreds of thousands of complex lines of code like traditional software. The code to build the model is often very short and linear (just a few loops of forward and backward functions). The complexity, intelligence, and problem-solving ability of the model are entirely distilled from the massive volume of data it absorbs during training.

## Parameters vs Hyperparameters
### 1. Distinguishing Parameters and Hyperparameters
In a neural network, two types of variables work together to create a predictive model. Clearly distinguishing them is a prerequisite for tuning the network.

- **Parameters: $W$ and $b$**

These are the internal variables of the model. The values of $W$ (weights) and $b$ (biases) are not directly set by human programmers. Instead, they are automatically learned and updated by the model through each iteration of the Gradient Descent algorithm based on the provided data.

- **Hyperparameters**

These are the external variables that you (the machine learning engineer) directly set before the training process begins. They act as coordinators, instructing the learning algorithm on how to operate, how fast or slow to learn, and on which architecture. These hyperparameters will directly shape the final set of parameters $W$ and $b$.

Basic hyperparameters include:
- $\alpha$ (Learning Rate): Determines the size of the weight update step in Gradient Descent.
- Number of iterations / epochs: How many rounds the training process will run.
- $L$: The number of hidden layers in the neural network.
- $n^{[l]}$: The number of neurons in each hidden layer.
- Activation Functions: Choosing to use ReLU, Sigmoid, or Tanh at the layers.

<img src="../images/Week04/image12.png" width=800>

### 2. The Empirical Nature of Deep Learning
A truth that often confuses beginners in Deep Learning is: There is no fixed mathematical formula that dictates the perfect hyperparameter configuration for a new problem in advance. Implementing Deep Learning is largely an empirical process that revolves around an iterative cycle: **Idea $\rightarrow$ Code $\rightarrow$ Experiment**.

- **Example of tuning the Learning Rate ($\alpha$):**

  You start with an educated guess, for example $\alpha = 0.01$. You run the model and plot the Cost Function $J$.

  - If the graph goes down too slowly: $\alpha$ is too small, the model learns sluggishly. You increase it to $0.05$.
  - If the graph fluctuates wildly or spikes upward: $\alpha$ is too large, the algorithm overshoots the optimal point. You decrease it to $0.001$.
  - You continue this trial-and-error process until you find a cost curve that slopes downward smoothly and converges the fastest.

- **Variations across time and domains:**

    The intuition about hyperparameters you gain from Computer Vision might not apply to Natural Language Processing (NLP) or online advertising data. Furthermore, even if you are solving the exact same problem, today's optimal configuration might become obsolete next year due to changes in the nature of the data or hardware infrastructure upgrades (CPU/GPU).

<img src="../images/Week04/image13.png" width=800>

### 3. Advice for Tuning
This reliance on empirical testing can sometimes feel scientifically unsatisfying. However, at present, it is the most effective method.

The advice for you is: Do not be afraid to make changes. When starting a new project, boldly experiment with a wide range of different hyperparameter values. Run the model, cross-reference the results on the Cross-Validation set, and select the combination that yields the best performance. Over time (months or years), periodically re-evaluate to see if a new set of hyperparameters performs better than the old one.
